# 🔋 Smart Energy Management System
## Fuzzy Expert System — Section 1

**Problem Statement:**  
Managing energy in a hybrid system (Solar + Battery + Grid) is complex due to uncertain and continuously changing conditions such as solar irradiance, battery charge level, and grid stability. A Fuzzy Expert System is designed to intelligently control:
- How much to depend on the grid
- How to control the load
- Whether to charge or discharge the battery
- Whether to curtail excess solar power
- The priority of battery charging

**Inputs:** Battery SoC, Solar Production, Grid Status, Cumulative Consumption, Current Demand  
**Outputs:** Grid Dependency, Load Control, Battery Action, Solar Curtailment, Battery Charging Priority

## 1. Install & Import Libraries

In [ ]:
!pip install scikit-fuzzy ipywidgets openpyxl arabic-reshaper python-bidi pandas-stubs -q

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

%matplotlib inline
print('✅ All libraries imported successfully!')

## 2. Define Universe of Discourse (Input & Output Ranges)

In [ ]:
soc_range         = np.arange(0, 101, 1)   # Battery SoC: 0-100%
solar_range       = np.arange(0, 101, 1)   # Solar Production: 0-100%
grid_range        = np.arange(0, 101, 1)   # Grid Status: 0-100%
consumption_range = np.arange(0, 1001, 1)  # Cumulative Consumption: 0-1000 kWh
load_range        = np.arange(0, 101, 1)   # Current Load Demand: 0-100%
output_range      = np.arange(0, 101, 1)   # All outputs: 0-100%

# --- Antecedents (Inputs) ---
battery_soc      = ctrl.Antecedent(soc_range, 'battery_soc')
solar_production = ctrl.Antecedent(solar_range, 'solar_production')
grid_status      = ctrl.Antecedent(grid_range, 'grid_status')
cum_consumption  = ctrl.Antecedent(consumption_range, 'cum_consumption')
current_demand   = ctrl.Antecedent(load_range, 'current_demand')

# --- Consequents (Outputs) ---
grid_dependency           = ctrl.Consequent(output_range, 'grid_dependency',           defuzzify_method='centroid')
load_control              = ctrl.Consequent(output_range, 'load_control',              defuzzify_method='centroid')
battery_action            = ctrl.Consequent(output_range, 'battery_action',            defuzzify_method='centroid')
solar_curtailment         = ctrl.Consequent(output_range, 'solar_curtailment',         defuzzify_method='centroid')
battery_charging_priority = ctrl.Consequent(output_range, 'battery_charging_priority', defuzzify_method='centroid')

print('✅ Variables defined!')

## 3. Membership Functions

In [ ]:
# --- Battery SoC ---
battery_soc['empty']  = fuzz.trapmf(soc_range, [0,  0,  15, 25])
battery_soc['low']    = fuzz.trimf(soc_range,  [20, 35, 50])
battery_soc['medium'] = fuzz.trimf(soc_range,  [45, 60, 75])
battery_soc['good']   = fuzz.trimf(soc_range,  [70, 80, 90])
battery_soc['full']   = fuzz.trapmf(soc_range, [85, 95, 100, 100])

# --- Solar Production ---
solar_production['none']   = fuzz.trapmf(solar_range, [0,  0,  5,  15])
solar_production['low']    = fuzz.trimf(solar_range,  [10, 25, 45])
solar_production['medium'] = fuzz.trimf(solar_range,  [35, 55, 75])
solar_production['high']   = fuzz.trapmf(solar_range, [65, 85, 100, 100])

# --- Grid Status ---
grid_status['off']       = fuzz.trapmf(grid_range, [0,  0,  5,  15])
grid_status['weak']      = fuzz.trimf(grid_range,  [10, 30, 50])
grid_status['stable']    = fuzz.trimf(grid_range,  [45, 70, 85])
grid_status['excellent'] = fuzz.trapmf(grid_range, [80, 90, 100, 100])

# --- Cumulative Consumption ---
cum_consumption['safe']     = fuzz.trapmf(consumption_range, [0,   0,   250, 300])
cum_consumption['warning']  = fuzz.trimf(consumption_range,  [250, 440, 490])
cum_consumption['border']   = fuzz.trimf(consumption_range,  [480, 550, 650])
cum_consumption['critical'] = fuzz.trapmf(consumption_range, [600, 750, 1000, 1000])

# --- Current Demand ---
current_demand['low']    = fuzz.trapmf(load_range, [0,  0,  15, 35])
current_demand['medium'] = fuzz.trimf(load_range,  [25, 50, 75])
current_demand['high']   = fuzz.trapmf(load_range, [65, 85, 100, 100])

# --- Grid Dependency (Output) ---
grid_dependency['islanded'] = fuzz.trapmf(output_range, [0,  0,  10, 20])
grid_dependency['low']      = fuzz.trimf(output_range,  [15, 30, 45])
grid_dependency['medium']   = fuzz.trimf(output_range,  [40, 50, 60])
grid_dependency['high']     = fuzz.trimf(output_range,  [55, 70, 85])
grid_dependency['full']     = fuzz.trapmf(output_range, [80, 90, 100, 100])

# --- Load Control (Output) ---
load_control['critical'] = fuzz.trapmf(output_range, [0,  0,  10, 20])
load_control['minimal']  = fuzz.trimf(output_range,  [15, 30, 45])
load_control['eco']      = fuzz.trimf(output_range,  [40, 50, 60])
load_control['normal']   = fuzz.trimf(output_range,  [55, 70, 85])
load_control['full']     = fuzz.trapmf(output_range, [80, 90, 100, 100])

# --- Battery Action (Output) ---
battery_action['fast_charge']    = fuzz.trapmf(output_range, [0,  0,  10, 20])
battery_action['charge']         = fuzz.trimf(output_range,  [15, 30, 45])
battery_action['idle']           = fuzz.trimf(output_range,  [40, 50, 60])
battery_action['eco_discharge']  = fuzz.trimf(output_range,  [55, 70, 85])
battery_action['full_discharge'] = fuzz.trapmf(output_range, [80, 90, 100, 100])

# --- Solar Curtailment (Output) ---
solar_curtailment['none']   = fuzz.trapmf(output_range, [0,  0,  10, 20])
solar_curtailment['low']    = fuzz.trimf(output_range,  [15, 30, 45])
solar_curtailment['medium'] = fuzz.trimf(output_range,  [40, 50, 60])
solar_curtailment['high']   = fuzz.trimf(output_range,  [55, 70, 85])
solar_curtailment['full']   = fuzz.trapmf(output_range, [80, 90, 100, 100])

# --- Battery Charging Priority (Output) ---
battery_charging_priority['none']   = fuzz.trapmf(output_range, [0,  0,  10, 20])
battery_charging_priority['low']    = fuzz.trimf(output_range,  [15, 30, 45])
battery_charging_priority['medium'] = fuzz.trimf(output_range,  [40, 50, 60])
battery_charging_priority['high']   = fuzz.trimf(output_range,  [55, 70, 85])
battery_charging_priority['urgent'] = fuzz.trapmf(output_range, [80, 90, 100, 100])

print('✅ Membership functions defined!')

## 4. Visualize Membership Functions

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
fig.suptitle('Membership Functions — Smart Energy Management FES', fontsize=16, fontweight='bold', y=1.01)

inputs = [
    (battery_soc,      'Battery SoC (%)',          ['empty','low','medium','good','full']),
    (solar_production, 'Solar Production (%)',      ['none','low','medium','high']),
    (grid_status,      'Grid Status (%)',           ['off','weak','stable','excellent']),
    (cum_consumption,  'Cumulative Consumption (kWh)', ['safe','warning','border','critical']),
    (current_demand,   'Current Demand (%)',        ['low','medium','high']),
]
outputs = [
    (grid_dependency,           'Grid Dependency (%)',           ['islanded','low','medium','high','full']),
    (load_control,              'Load Control (%)',              ['critical','minimal','eco','normal','full']),
    (battery_action,            'Battery Action (%)',            ['fast_charge','charge','idle','eco_discharge','full_discharge']),
    (solar_curtailment,         'Solar Curtailment (%)',         ['none','low','medium','high','full']),
    (battery_charging_priority, 'Battery Charging Priority (%)', ['none','low','medium','high','urgent']),
]

colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6']

for i, (var, title, terms) in enumerate(inputs):
    ax = axes[i][0]
    for j, term in enumerate(terms):
        ax.plot(var.universe, var[term].mf, label=term, color=colors[j], linewidth=2)
    ax.set_title(f'Input: {title}', fontweight='bold')
    ax.set_xlabel(title); ax.set_ylabel('Membership')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.1)

for i, (var, title, terms) in enumerate(outputs):
    ax = axes[i][1]
    for j, term in enumerate(terms):
        ax.plot(var.universe, var[term].mf, label=term, color=colors[j], linewidth=2)
    ax.set_title(f'Output: {title}', fontweight='bold')
    ax.set_xlabel(title); ax.set_ylabel('Membership')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig('membership_functions.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Membership functions plotted!')

## 5. Rule Base (30 Rules)

Rules are designed following energy management best practices:
- When battery is empty → fast charge, reduce load, maximize grid/solar use
- When solar is high + battery full → curtail solar, reduce grid dependency
- When grid is off → go islanded, discharge battery
- When consumption is critical → shed load aggressively
- etc.

In [ ]:
rules = [
    # --- Emergency: Battery Empty ---
    ctrl.Rule(
        battery_soc['empty'] & grid_status['excellent'],
        [grid_dependency['full'], battery_action['fast_charge'],
         load_control['eco'], battery_charging_priority['urgent'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        battery_soc['empty'] & grid_status['off'] & solar_production['high'],
        [grid_dependency['islanded'], battery_action['fast_charge'],
         load_control['minimal'], battery_charging_priority['urgent'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        battery_soc['empty'] & grid_status['off'] & solar_production['none'],
        [grid_dependency['islanded'], battery_action['fast_charge'],
         load_control['critical'], battery_charging_priority['urgent'], solar_curtailment['none']]
    ),

    # --- Low Battery ---
    ctrl.Rule(
        battery_soc['low'] & solar_production['high'],
        [grid_dependency['low'], battery_action['charge'],
         load_control['eco'], battery_charging_priority['high'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        battery_soc['low'] & solar_production['none'] & grid_status['excellent'],
        [grid_dependency['high'], battery_action['charge'],
         load_control['normal'], battery_charging_priority['high'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        battery_soc['low'] & current_demand['high'],
        [grid_dependency['high'], battery_action['eco_discharge'],
         load_control['eco'], battery_charging_priority['medium'], solar_curtailment['none']]
    ),

    # --- Medium Battery ---
    ctrl.Rule(
        battery_soc['medium'] & solar_production['medium'] & grid_status['stable'],
        [grid_dependency['medium'], battery_action['idle'],
         load_control['normal'], battery_charging_priority['medium'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        battery_soc['medium'] & current_demand['low'],
        [grid_dependency['low'], battery_action['charge'],
         load_control['full'], battery_charging_priority['medium'], solar_curtailment['low']]
    ),
    ctrl.Rule(
        battery_soc['medium'] & grid_status['off'],
        [grid_dependency['islanded'], battery_action['eco_discharge'],
         load_control['eco'], battery_charging_priority['low'], solar_curtailment['none']]
    ),

    # --- Good Battery ---
    ctrl.Rule(
        battery_soc['good'] & solar_production['high'] & current_demand['low'],
        [grid_dependency['islanded'], battery_action['idle'],
         load_control['full'], battery_charging_priority['low'], solar_curtailment['medium']]
    ),
    ctrl.Rule(
        battery_soc['good'] & current_demand['high'],
        [grid_dependency['medium'], battery_action['eco_discharge'],
         load_control['full'], battery_charging_priority['low'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        battery_soc['good'] & grid_status['excellent'],
        [grid_dependency['medium'], battery_action['idle'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['low']]
    ),

    # --- Full Battery ---
    ctrl.Rule(
        battery_soc['full'] & solar_production['high'],
        [grid_dependency['islanded'], battery_action['idle'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['full']]
    ),
    ctrl.Rule(
        battery_soc['full'] & current_demand['high'],
        [grid_dependency['low'], battery_action['full_discharge'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['low']]
    ),
    ctrl.Rule(
        battery_soc['full'] & current_demand['low'] & solar_production['medium'],
        [grid_dependency['islanded'], battery_action['idle'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['high']]
    ),

    # --- Grid Off Scenarios ---
    ctrl.Rule(
        grid_status['off'] & battery_soc['good'] & solar_production['high'],
        [grid_dependency['islanded'], battery_action['idle'],
         load_control['full'], battery_charging_priority['low'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        grid_status['off'] & battery_soc['medium'] & solar_production['low'],
        [grid_dependency['islanded'], battery_action['eco_discharge'],
         load_control['minimal'], battery_charging_priority['low'], solar_curtailment['none']]
    ),

    # --- Critical Consumption ---
    ctrl.Rule(
        cum_consumption['critical'] & current_demand['high'],
        [grid_dependency['low'], battery_action['idle'],
         load_control['critical'], battery_charging_priority['none'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        cum_consumption['critical'] & battery_soc['good'],
        [grid_dependency['low'], battery_action['eco_discharge'],
         load_control['minimal'], battery_charging_priority['none'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        cum_consumption['border'] & current_demand['medium'],
        [grid_dependency['medium'], battery_action['idle'],
         load_control['eco'], battery_charging_priority['low'], solar_curtailment['none']]
    ),

    # --- Solar Surplus Scenarios ---
    ctrl.Rule(
        solar_production['high'] & battery_soc['medium'] & current_demand['low'],
        [grid_dependency['islanded'], battery_action['charge'],
         load_control['full'], battery_charging_priority['high'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        solar_production['high'] & battery_soc['full'] & grid_status['excellent'],
        [grid_dependency['islanded'], battery_action['idle'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['full']]
    ),

    # --- Grid Weak ---
    ctrl.Rule(
        grid_status['weak'] & battery_soc['low'],
        [grid_dependency['low'], battery_action['charge'],
         load_control['minimal'], battery_charging_priority['medium'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        grid_status['weak'] & solar_production['medium'],
        [grid_dependency['low'], battery_action['charge'],
         load_control['eco'], battery_charging_priority['medium'], solar_curtailment['none']]
    ),

    # --- Normal Operation ---
    ctrl.Rule(
        grid_status['stable'] & battery_soc['medium'] & solar_production['medium'] & current_demand['medium'],
        [grid_dependency['medium'], battery_action['idle'],
         load_control['normal'], battery_charging_priority['medium'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        grid_status['excellent'] & battery_soc['good'] & cum_consumption['safe'],
        [grid_dependency['medium'], battery_action['idle'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['none']]
    ),

    # --- High Demand Edge Cases ---
    ctrl.Rule(
        current_demand['high'] & grid_status['excellent'] & battery_soc['good'],
        [grid_dependency['high'], battery_action['eco_discharge'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['none']]
    ),
    ctrl.Rule(
        current_demand['high'] & grid_status['off'] & battery_soc['empty'],
        [grid_dependency['islanded'], battery_action['fast_charge'],
         load_control['critical'], battery_charging_priority['urgent'], solar_curtailment['none']]
    ),

    # --- Safe Consumption + Good Resources ---
    ctrl.Rule(
        cum_consumption['safe'] & solar_production['high'] & battery_soc['good'],
        [grid_dependency['islanded'], battery_action['idle'],
         load_control['full'], battery_charging_priority['none'], solar_curtailment['medium']]
    ),
    ctrl.Rule(
        cum_consumption['warning'] & grid_status['stable'] & current_demand['medium'],
        [grid_dependency['medium'], battery_action['idle'],
         load_control['eco'], battery_charging_priority['low'], solar_curtailment['none']]
    ),
]

print(f'✅ {len(rules)} rules defined!')

## 6. Build Inference Engine (Mamdani) + Defuzzification (Centroid)

In [ ]:
energy_ctrl = ctrl.ControlSystem(rules)
energy_sim  = ctrl.ControlSystemSimulation(energy_ctrl)

print('✅ Mamdani Inference Engine built with Centroid Defuzzification!')
print('   Method: Mamdani (min-inference, max-aggregation)')
print('   Defuzzification: Centroid of Area (CoA)')

## 7. Interactive User Interface (ipywidgets)

In [ ]:
def run_fuzzy_system(soc, solar, grid, consumption, demand):
    """Run the fuzzy system and return results dict."""
    sim = ctrl.ControlSystemSimulation(energy_ctrl)
    sim.input['battery_soc']      = soc
    sim.input['solar_production'] = solar
    sim.input['grid_status']      = grid
    sim.input['cum_consumption']  = consumption
    sim.input['current_demand']   = demand
    sim.compute()
    return {
        'Grid Dependency (%)':           round(sim.output['grid_dependency'], 2),
        'Load Control (%)':              round(sim.output['load_control'], 2),
        'Battery Action (%)':            round(sim.output['battery_action'], 2),
        'Solar Curtailment (%)':         round(sim.output['solar_curtailment'], 2),
        'Battery Charging Priority (%)': round(sim.output['battery_charging_priority'], 2),
    }

def label_output(key, val):
    """Map output score to human-readable label."""
    labels = {
        'Grid Dependency (%)':           [(12,'Islanded'),(30,'Low'),(50,'Medium'),(70,'High'),(100,'Full')],
        'Load Control (%)':              [(12,'Critical'),(30,'Minimal'),(50,'Eco'),(70,'Normal'),(100,'Full')],
        'Battery Action (%)':            [(12,'Fast Charge'),(30,'Charge'),(50,'Idle'),(70,'Eco Discharge'),(100,'Full Discharge')],
        'Solar Curtailment (%)':         [(12,'None'),(30,'Low'),(50,'Medium'),(70,'High'),(100,'Full')],
        'Battery Charging Priority (%)': [(12,'None'),(30,'Low'),(50,'Medium'),(70,'High'),(100,'Urgent')],
    }
    for threshold, lbl in labels.get(key, []):
        if val <= threshold:
            return lbl
    return 'Full'

def color_bar(val, color='#3498db'):
    pct = val
    return f'''
    <div style="background:#ecf0f1;border-radius:8px;height:18px;width:100%;margin-top:4px">
      <div style="background:{color};width:{pct}%;height:18px;border-radius:8px;"></div>
    </div>'''

output_colors = ['#e74c3c','#e67e22','#2ecc71','#f39c12','#9b59b6']

def on_compute(btn):
    out_area.clear_output()
    with out_area:
        try:
            results = run_fuzzy_system(
                soc_slider.value, solar_slider.value, grid_slider.value,
                consumption_slider.value, demand_slider.value
            )
            html = '''
            <div style="font-family:Arial;padding:16px;background:#1a1a2e;border-radius:12px;color:white">
              <h3 style="text-align:center;color:#3498db;margin-bottom:20px">🔋 FES Output Results</h3>
            '''
            for i, (k, v) in enumerate(results.items()):
                lbl = label_output(k, v)
                bar = color_bar(v, output_colors[i % len(output_colors)])
                html += f'''
                <div style="margin-bottom:14px;padding:10px;background:#16213e;border-radius:8px;">
                  <b style="color:#ecf0f1">{k}</b>
                  <span style="float:right;color:{output_colors[i%len(output_colors)]};font-size:1.1em">
                    {v}% — <i>{lbl}</i>
                  </span>
                  {bar}
                </div>'''
            html += '</div>'
            display(HTML(html))

            # Plot outputs
            fig, axes = plt.subplots(1, 5, figsize=(20, 3))
            output_vars = [grid_dependency, load_control, battery_action, solar_curtailment, battery_charging_priority]
            result_vals  = list(results.values())
            result_keys  = list(results.keys())
            for i, (var, val, key) in enumerate(zip(output_vars, result_vals, result_keys)):
                ax = axes[i]
                for term in var.terms:
                    ax.plot(var.universe, var[term].mf, linewidth=1.5, label=term)
                ax.axvline(x=val, color='red', linestyle='--', linewidth=2, label=f'Output={val}')
                ax.set_title(key.replace(' (%)', ''), fontsize=9, fontweight='bold')
                ax.set_ylim(-0.05, 1.1); ax.grid(True, alpha=0.3); ax.legend(fontsize=6)
            plt.suptitle('Defuzzified Outputs (red line = crisp value)', fontweight='bold')
            plt.tight_layout(); plt.show()

        except Exception as e:
            display(HTML(f'<div style="color:red;padding:10px">⚠️ Error: {str(e)}<br>Try adjusting input sliders.</div>'))

# Build UI
style   = {'description_width': '220px'}
layout  = widgets.Layout(width='600px')

soc_slider         = widgets.IntSlider(value=40,  min=0,   max=100,  step=1,  description='🔋 Battery SoC (%):',           style=style, layout=layout)
solar_slider       = widgets.IntSlider(value=60,  min=0,   max=100,  step=1,  description='☀️ Solar Production (%):',      style=style, layout=layout)
grid_slider        = widgets.IntSlider(value=70,  min=0,   max=100,  step=1,  description='⚡ Grid Status (%):',           style=style, layout=layout)
consumption_slider = widgets.IntSlider(value=300, min=0,   max=1000, step=10, description='📊 Cumulative Consumption (kWh):',style=style, layout=layout)
demand_slider      = widgets.IntSlider(value=50,  min=0,   max=100,  step=1,  description='🔌 Current Demand (%):',        style=style, layout=layout)

btn = widgets.Button(
    description='⚙️  Run Fuzzy System',
    button_style='success',
    layout=widgets.Layout(width='250px', height='40px', margin='10px 0')
)
btn.on_click(on_compute)

out_area = widgets.Output()

header = HTML('''
<div style="font-family:Arial;background:linear-gradient(135deg,#1a1a2e,#16213e);padding:20px;border-radius:12px;margin-bottom:15px">
  <h2 style="color:#3498db;margin:0">🔋 Smart Energy Management — Fuzzy Expert System</h2>
  <p style="color:#bdc3c7;margin:5px 0 0">Adjust the sliders and click Run to see the system's decisions.</p>
</div>
''')

ui = widgets.VBox([
    header,
    soc_slider, solar_slider, grid_slider, consumption_slider, demand_slider,
    btn, out_area
])
display(ui)

## 8. Testing & Validation

We test the system on **10 predefined scenarios** representing different real-world energy states.

In [ ]:
test_cases = [
    # (SoC, Solar, Grid, Consumption, Demand, Scenario)
    (10,  0,  0,  800, 90, 'CRITICAL: Empty battery, no solar, grid off, high demand'),
    (10,  90, 50, 200, 30, 'Emergency battery, high solar, stable grid, low demand'),
    (50,  60, 75, 300, 50, 'Normal operation: medium everything'),
    (90,  95, 90, 150, 20, 'Ideal: full battery, high solar, excellent grid, low demand'),
    (30,  0,  95, 500, 70, 'Low battery, no solar, excellent grid, border consumption'),
    (80,  80, 0,  250, 60, 'Good battery, high solar, grid off'),
    (60,  10, 60, 700, 80, 'Medium battery, low solar, critical consumption, high demand'),
    (20,  50, 30, 400, 40, 'Low battery, medium solar, weak grid'),
    (70,  30, 85, 350, 55, 'Good battery, low solar, stable grid'),
    (45,  70, 70, 480, 35, 'Medium battery, high solar, stable grid, border consumption'),
]

results_list = []
print('Running test scenarios...\n')
for soc, solar, grid, cons, demand, scenario in test_cases:
    try:
        r = run_fuzzy_system(soc, solar, grid, cons, demand)
        row = {'Scenario': scenario[:50], 'SoC': soc, 'Solar': solar,
               'Grid': grid, 'Consumption': cons, 'Demand': demand}
        row.update(r)
        results_list.append(row)
        print(f'✅ {scenario[:55]}')
    except Exception as e:
        print(f'⚠️  {scenario[:50]} — Error: {e}')

df_results = pd.DataFrame(results_list)
print('\n✅ All tests completed!')
df_results

In [ ]:
# --- Visualize Test Results ---
output_cols = ['Grid Dependency (%)', 'Load Control (%)', 'Battery Action (%)',
               'Solar Curtailment (%)', 'Battery Charging Priority (%)']

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
colors_test = ['#e74c3c','#e67e22','#2ecc71','#f39c12','#9b59b6']

for i, (col, color) in enumerate(zip(output_cols, colors_test)):
    ax = axes[i]
    ax.barh(range(len(df_results)), df_results[col], color=color, alpha=0.8)
    ax.set_yticks(range(len(df_results)))
    ax.set_yticklabels([f'T{j+1}' for j in range(len(df_results))], fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_title(col.replace(' (%)', ''), fontweight='bold', fontsize=9)
    ax.set_xlabel('%'); ax.grid(True, axis='x', alpha=0.3)
    for j, v in enumerate(df_results[col]):
        ax.text(v+1, j, f'{v:.0f}', va='center', fontsize=8)

plt.suptitle('Test Results — All 10 Scenarios × 5 Outputs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('test_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Test validation chart saved!')

In [ ]:
# --- Export Results to Excel ---
df_results.to_excel('fuzzy_test_results.xlsx', index=False)
print('✅ Results exported to fuzzy_test_results.xlsx')

# --- Summary Statistics ---
print('\n📊 Summary Statistics:')
display(df_results[output_cols].describe().round(2))

## 9. System Summary

| Component | Details |
|---|---|
| **Problem** | Smart Energy Management (Solar + Battery + Grid) |
| **Inputs** | Battery SoC, Solar Production, Grid Status, Cumulative Consumption, Current Demand |
| **Outputs** | Grid Dependency, Load Control, Battery Action, Solar Curtailment, Battery Charging Priority |
| **Membership Functions** | Trapezoidal (trapmf) + Triangular (trimf) |
| **Rules** | 30 expert-designed rules |
| **Inference** | Mamdani (min-inference, max-aggregation) |
| **Defuzzification** | Centroid of Area (CoA) |
| **UI** | Interactive ipywidgets sliders + live plots |
| **Testing** | 10 real-world scenarios + Excel export |

### References
1. Zadeh, L.A. (1965). *Fuzzy sets*. Information and Control, 8(3), 338–353.
2. Mamdani, E.H. (1974). *Application of fuzzy algorithms for control of simple dynamic plant*. Proc. IEE, 121(12), 1585–1588.
3. scikit-fuzzy documentation: https://pythonhosted.org/scikit-fuzzy/
4. Sugeno, M. (1985). *Industrial applications of fuzzy control*. Elsevier.
5. Kamankesh, H. et al. (2016). *Optimal scheduling of renewable micro-grids considering plug-in hybrid electric vehicle charging demand*. Energy, 100, 285–297.